In [18]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../data/raw/ecommerce.db")

Load documents

In [19]:
customers = pd.read_sql("SELECT * FROM customers", conn)
products = pd.read_sql("SELECT * FROM products", conn)
orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)
reviews = pd.read_sql("SELECT * FROM reviews", conn)
web_sessions = pd.read_sql("SELECT * FROM web_sessions", conn)

In [22]:
legacy = pd.read_csv("../data/raw/legacy_customers_export.csv")
catalog = pd.read_csv("../data/raw/product_catalog_2024.csv")

In [23]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Reviews:", reviews.shape)
print("Web sessions:", web_sessions.shape)
print("Legacy customers:", legacy.shape)
print("Product catalog:", catalog.shape)

Customers: (2500, 8)
Products: (300, 6)
Orders: (9000, 5)
Order items: (20362, 6)
Reviews: (4000, 6)
Web sessions: (12000, 6)
Legacy customers: (1427, 5)
Product catalog: (267, 6)


In [24]:
customers.head()

,customer_id,name,email,signup_date,city,country,age,gender
0,1,Allison Hill,donaldgarcia@example.net,2019-08-17,Sydney,Australia,63.0,M
1,2,Angie Henderson,davisjesse@example.net,2023-02-25,Sydney,Australia,21.0,F
2,3,Cristian Santos,lrobinson@example.com,2021-10-31,Toronto,Canada,17.0,M
3,4,Abigail Shaffer,jpeterson@example.org,2020-03-27,Manchester,UK,53.0,M
4,5,Gabrielle Davis,howardmaurice@example.com,2022-11-30,Chicago,USA,37.0,M


### Legacy Customer Dates

In [25]:
legacy["Signup_Dt"].head(20)

0         30-Nov-2023
1         01-Aug-2021
2         27-May-2021
3     August 19, 2020
4          07/05/2023
5          01/25/2021
6     August 23, 2023
7        May 05, 2022
8          04/01/2024
9         23-Jan-2023
10         2019-09-25
11         2023-05-19
12        11-Jun-2021
13        10-Feb-2020
14      June 27, 2024
15         2022-12-23
16    August 31, 2022
17      June 06, 2020
18        11-Apr-2024
19         03/31/2023
Name: Signup_Dt, dtype: str

In [26]:
legacy["Signup_Dt"].dtype

<StringDtype(storage='python', na_value=nan)>

In [27]:
legacy["Signup_Dt"].dropna().astype(str).head(50).tolist()

['30-Nov-2023',
 '01-Aug-2021',
 '27-May-2021',
 'August 19, 2020',
 '07/05/2023',
 '01/25/2021',
 'August 23, 2023',
 'May 05, 2022',
 '04/01/2024',
 '23-Jan-2023',
 '2019-09-25',
 '2023-05-19',
 '11-Jun-2021',
 '10-Feb-2020',
 'June 27, 2024',
 '2022-12-23',
 'August 31, 2022',
 'June 06, 2020',
 '11-Apr-2024',
 '03/31/2023',
 '13-May-2023',
 '05/14/2020',
 '03-Aug-2019',
 '2024-04-12',
 '09/09/2020',
 '14-Dec-2019',
 '2019-12-27',
 'August 24, 2023',
 '2020-01-06',
 '27-May-2021',
 '29-Dec-2019',
 '04/24/2023',
 '01/13/2022',
 '2021-03-15',
 '05/24/2024',
 '18-Apr-2021',
 '10/04/2021',
 '11-Dec-2021',
 '16-Apr-2023',
 'April 28, 2021',
 '2020-03-15',
 'May 14, 2021',
 'October 14, 2022',
 'October 14, 2022',
 '05/29/2019',
 '03/26/2023',
 '2022-01-31',
 '2024-01-16',
 '2021-04-18',
 'November 21, 2021']

In [28]:
legacy["Signup_Dt"] = pd.to_datetime(
    legacy["Signup_Dt"],
    format="mixed",
    errors="coerce"
)

In [29]:
legacy["Signup_Dt"].dtype

dtype('<M8[us]')

In [30]:
legacy["Signup_Dt"].head(20)

0    2023-11-30
1    2021-08-01
2    2021-05-27
3    2020-08-19
4    2023-07-05
5    2021-01-25
6    2023-08-23
7    2022-05-05
8    2024-04-01
9    2023-01-23
10   2019-09-25
11   2023-05-19
12   2021-06-11
13   2020-02-10
14   2024-06-27
15   2022-12-23
16   2022-08-31
17   2020-06-06
18   2024-04-11
19   2023-03-31
Name: Signup_Dt, dtype: datetime64[us]

In [31]:
legacy["Signup_Dt"].describe()

count                          1426
mean     2021-08-30 21:23:28.695652
min             1900-01-01 00:00:00
25%             2020-05-14 00:00:00
50%             2021-09-28 00:00:00
75%             2023-01-29 18:00:00
max             2024-07-11 00:00:00
Name: Signup_Dt, dtype: object

In [32]:
legacy[["Signup_Dt"]].head(20)

,Signup_Dt
0,2023-11-30
1,2021-08-01
2,2021-05-27
3,2020-08-19
4,2023-07-05
5,2021-01-25
6,2023-08-23
7,2022-05-05
8,2024-04-01
9,2023-01-23


In [33]:
print("Date dtype:", legacy["Signup_Dt"].dtype)
print("Missing/unparseable dates:", legacy["Signup_Dt"].isna().sum())

Date dtype: datetime64[us]
Missing/unparseable dates: 1


### Rename the legacy columns

In [34]:
legacy = legacy.rename(columns={
    "Customer Name": "name",
    "EMAIL_ADDR": "email",
    "Signup_Dt": "signup_date",
    "Home City": "city",
    "Marketing Segment": "marketing_segment"
})

In [35]:
legacy.columns

Index(['name', 'email', 'signup_date', 'city', 'marketing_segment'], dtype='str')

### Standardize customer text

In [36]:
legacy["email"] = (
    legacy["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

customers["email"] = (
    customers["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [37]:
legacy["name"] = (
    legacy["name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

customers["name"] = (
    customers["name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

In [38]:
legacy["name"].head()

0     sarah robinson
1        donald todd
2    nathaniel floyd
3        sean carter
4    samuel williams
Name: name, dtype: string

### Customer duplicates

In [39]:
legacy.duplicated().sum()

np.int64(0)

In [40]:
customers.duplicated().sum()

np.int64(0)

### Near-duplicates using email

In [41]:
customer_email_map = customers[
    ["customer_id", "email"]
].dropna(subset=["email"]).drop_duplicates("email")

In [42]:
legacy_matched = legacy.merge(
    customer_email_map,
    on="email",
    how="left",
    indicator=True
)

In [43]:
legacy_matched["_merge"].value_counts()

_merge
both          1369
left_only       58
right_only       0
Name: count, dtype: int64

In [44]:
customers["age_missing"] = customers["age"].isna()
customers["city_missing"] = customers["city"].isna()
customers["gender_missing"] = customers["gender"].isna()

reviews["review_text_missing"] = reviews["review_text"].isna()

In [45]:
print("Missing age:", customers["age_missing"].sum())
print("Missing city:", customers["city_missing"].sum())
print("Missing gender:", customers["gender_missing"].sum())
print("Missing review text:", reviews["review_text_missing"].sum())

Missing age: 139
Missing city: 63
Missing gender: 115
Missing review text: 838


### Product price outliers

In [46]:
Q1 = products["unit_price"].quantile(0.25)
Q3 = products["unit_price"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 192.33249999999998
Q3: 579.5425
IQR: 387.21000000000004
Lower bound: -388.4825000000001
Upper bound: 1160.3575


In [47]:
price_outliers = products[
    (products["unit_price"] < lower_bound) |
    (products["unit_price"] > upper_bound)
]

price_outliers

,product_id,name,category,subcategory,unit_price,cost
244,245,Throughout Personal Care,Beauty & Health,Personal Care,11506.5,104.23
249,250,Discover Haircare,Beauty & Health,Haircare,8284.0,112.79
269,270,Environmental Non-Fiction,Books & Media,Non-Fiction,34872.0,252.13


In [48]:
print("Price outliers:", len(price_outliers))

Price outliers: 3


### Product catalog integration

In [49]:
catalog = catalog.rename(columns={
    "SKU": "product_id",
    "item_name": "name",
    "dept": "category",
    "list_price_usd": "unit_price",
    "supplier_cost": "cost",
    "in_stock_units": "stock"
})

In [50]:
catalog["product_id"] = (
    catalog["product_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

products["product_id"] = (
    products["product_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [51]:
product_merged = products.merge(
    catalog,
    on="product_id",
    how="outer",
    suffixes=("_db", "_catalog"),
    indicator=True
)

### Explicitly report source-only SKUs

In [52]:
database_only = product_merged[
    product_merged["_merge"] == "left_only"
]

database_only[["product_id", "name_db"]]

,product_id,name_db
9,107,Appear Women
11,109,Health Footwear
25,121,Themselves Footwear
26,122,Attorney Men
28,124,Act Kid
33,129,Direction Kid
34,13,Support Headphone
46,140,Fall Footwear
57,150,Population Kid
67,16,Yes Smartphone


In [53]:
catalog_only = product_merged[
    product_merged["_merge"] == "right_only"
]

catalog_only[["product_id", "name_catalog"]]

,product_id,name_catalog
291,9000,Small Prototype
292,9001,Standard Prototype
293,9002,Executive Prototype
294,9003,Task Prototype
295,9004,Growth Prototype
296,9005,Suddenly Prototype
297,9006,Individual Prototype
298,9007,Leader Prototype
299,9008,Sign Prototype
300,9009,Trial Prototype


In [54]:
print("Database only:", len(database_only))
print("Catalog only:", len(catalog_only))

Database only: 45
Catalog only: 12


### Remove exact duplicate order_items

In [55]:
order_items_before = len(order_items)

order_items = order_items.drop_duplicates()

order_items_after = len(order_items)

print("Before:", order_items_before)
print("After:", order_items_after)
print("Duplicates removed:", order_items_before - order_items_after)

Before: 20362
After: 20362
Duplicates removed: 0


### Treat negative quantities as returns

In [56]:
order_items["is_return"] = order_items["quantity"] < 0

In [57]:
order_items["revenue"] = (
    order_items["quantity"]
    * order_items["unit_price"]
)

In [58]:
order_items["is_return"]

0        False
1         True
2        False
3        False
4        False
         ...  
20357    False
20358    False
20359    False
20360    False
20361    False
Name: is_return, Length: 20362, dtype: bool

In [59]:
order_items["revenue"]

0          17.07
1        -133.29
2         560.24
3         216.27
4        1553.40
          ...   
20357    2114.25
20358      16.20
20359     697.79
20360     457.70
20361     148.68
Name: revenue, Length: 20362, dtype: float64

### Reshape using pivot_table

Category x month revenue

In [60]:
products["product_id"] = products["product_id"].astype(int)

In [61]:
sales = order_items.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

sales = sales.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

In [62]:
sales.head(20)

,order_item_id,order_id,product_id,quantity,unit_price,discount,is_return,revenue,order_date,category
0,1,1,272,3,5.69,0.15,False,17.07,2022-12-07 09:41:23,Books & Media
1,2,1,300,-1,133.29,0.15,True,-133.29,2022-12-07 09:41:23,Books & Media
2,3,1,273,1,560.24,0.00,False,560.24,2022-12-07 09:41:23,Books & Media
3,4,2,11,1,216.27,0.00,False,216.27,2024-06-30 10:09:52,Electronics
4,5,2,239,3,517.80,0.15,False,1553.40,2024-06-30 10:09:52,Beauty & Health
5,6,3,58,1,87.55,0.00,False,87.55,2022-04-21 08:21:52,Home & Kitchen
6,7,3,70,1,402.01,0.00,False,402.01,2022-04-21 08:21:52,Home & Kitchen
7,8,3,65,1,460.75,0.00,False,460.75,2022-04-21 08:21:52,Home & Kitchen
8,9,3,279,2,226.81,0.00,False,453.62,2022-04-21 08:21:52,Books & Media
9,10,3,174,1,864.84,0.00,False,864.84,2022-04-21 08:21:52,Sports & Outdoors


In [63]:
sales["product_id"].dtype

dtype('int64')

In [64]:
sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    errors="coerce"
)

In [65]:
sales["order_date"].dtype

dtype('<M8[us]')

In [66]:
sales["revenue"] = (
    sales["quantity"] * sales["unit_price"]
)

In [67]:
sales["revenue"]

0          17.07
1        -133.29
2         560.24
3         216.27
4        1553.40
          ...   
20357    2114.25
20358      16.20
20359     697.79
20360     457.70
20361     148.68
Name: revenue, Length: 20362, dtype: float64

In [68]:
sales.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,is_return,revenue,order_date,category
0,1,1,272,3,5.69,0.15,False,17.07,2022-12-07 09:41:23,Books & Media
1,2,1,300,-1,133.29,0.15,True,-133.29,2022-12-07 09:41:23,Books & Media
2,3,1,273,1,560.24,0.00,False,560.24,2022-12-07 09:41:23,Books & Media
3,4,2,11,1,216.27,0.00,False,216.27,2024-06-30 10:09:52,Electronics
4,5,2,239,3,517.80,0.15,False,1553.40,2024-06-30 10:09:52,Beauty & Health


In [69]:
sales["month"] = sales["order_date"].dt.to_period("M")

In [70]:
sales["month"].head()

0    2022-12
1    2022-12
2    2022-12
3    2024-06
4    2024-06
Name: month, dtype: period[M]

In [71]:
products.columns.tolist()

['product_id', 'name', 'category', 'subcategory', 'unit_price', 'cost']

In [72]:
category_month_revenue = sales.pivot_table(
    index="category",
    columns="month",
    values="revenue",
    aggfunc="sum",
    fill_value=0
)

In [73]:
category_month_revenue

month,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,...,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
category,,,,,,,,,,,,,,,,,,,,,
Apparel,21704.41,27139.78,29143.14,34768.97,32486.68,34065.76,33807.95,26067.20,29658.82,39100.95,...,70738.25,55492.75,69483.51,68306.90,75853.08,81726.06,70669.13,66184.44,64450.93,72875.04
Beauty & Health,63893.32,43159.97,19005.64,53134.58,72014.82,50183.65,106245.74,103567.50,68631.12,86271.21,...,77401.04,68900.15,144327.20,78919.90,110361.37,103980.89,103523.75,98010.91,194540.86,128320.98
Books & Media,129759.85,27406.75,92696.52,21060.67,96721.34,29311.82,135716.78,-10265.35,61982.28,102393.04,...,154299.27,162468.44,132289.23,195913.75,139125.27,207742.90,170881.50,99075.49,175017.04,101068.63
Electronics,27051.85,19576.59,31043.63,33880.36,39856.87,24977.35,29751.91,38838.08,40155.43,31688.34,...,51090.49,56569.44,62287.59,60349.81,70053.84,69871.75,73297.89,68849.34,79699.52,82424.02
Home & Kitchen,27848.95,21349.87,26076.30,15341.77,17698.95,27693.33,29178.91,30891.15,30586.90,32921.56,...,45152.82,44184.65,67762.51,50375.33,69480.31,73043.86,46226.27,66349.64,56212.60,56461.76
Sports & Outdoors,30485.32,29328.06,34092.16,20619.38,37621.22,36850.03,29963.27,38463.08,28636.52,33405.80,...,57865.82,64184.22,64221.26,64125.40,81582.09,81382.25,56585.39,69531.98,70499.77,65053.10


### Time-series analysis

In [74]:
monthly_revenue = (
    sales
    .set_index("order_date")
    .resample("ME")["revenue"]
    .sum()
)

In [75]:
monthly_revenue.head()

order_date
2022-01-31    300743.70
2022-02-28    167961.02
2022-03-31    232057.39
2022-04-30    178805.73
2022-05-31    296399.88
Freq: ME, Name: revenue, dtype: float64

In [77]:
customers.to_csv(
    "../data/processed/customers_clean.csv",
    index=False
)

In [78]:
products.to_csv(
    "../data/processed/products_clean.csv",
    index=False
)

In [79]:
orders.to_csv(
    "../data/processed/orders_clean.csv",
    index=False
)

In [80]:
order_items.to_csv(
    "../data/processed/order_items_clean.csv",
    index=False
)

In [81]:
reviews.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)

In [82]:
legacy.to_csv(
    "../data/processed/legacy_customers_clean.csv",
    index=False
)

In [83]:
catalog.to_csv(
    "../data/processed/product_catalog_clean.csv",
    index=False
)

In [84]:
sales.to_csv(
    "../data/processed/sales_clean.csv",
    index=False
)

#### Phase 3 roadmap

have 4 required routines: <br>

1. RFM Segmentation
- Recency
- Frequency
- Monetary
- NumPy-based scoring
2. Similarity / Recommendation
- Cosine similarity using NumPy
- Recommend 3 products for 5 customers
3. Regression
- Normal equation
- Manual R^2
4. Monte Carlo
- ≥5,000 trials
- 3 products
- Probability of stockout
- Confidence interval

#### for each one:

Mathemaatical formula <br>
| <br>
Explanation <br>
| <br>
Numpy implementation <br>
| <br>
Result <br>
| <br>
Sanity check <br>

#### 1. RFM Segmentation

In [85]:
sales.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,is_return,revenue,order_date,category,month
0,1,1,272,3,5.69,0.15,False,17.07,2022-12-07 09:41:23,Books & Media,2022-12
1,2,1,300,-1,133.29,0.15,True,-133.29,2022-12-07 09:41:23,Books & Media,2022-12
2,3,1,273,1,560.24,0.00,False,560.24,2022-12-07 09:41:23,Books & Media,2022-12
3,4,2,11,1,216.27,0.00,False,216.27,2024-06-30 10:09:52,Electronics,2024-06
4,5,2,239,3,517.80,0.15,False,1553.40,2024-06-30 10:09:52,Beauty & Health,2024-06


In [86]:
sales = sales.merge(
    orders[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)

### RFM formular

Recency <br>

#### $ R_{i} = D_{analysis} - D_{lastPurchase,i}$

Frequency

#### $ F_{i} =$ number of unique orders by customer $i  $

Monetary
#### $M_{i} = \sum$ net revenue for customer $i$

Calculate each customer's RFM values

In [87]:
import numpy as np

customer_ids = sales["customer_id"].to_numpy()

unique_customers = np.unique(customer_ids)

print("Number of customers:", len(unique_customers))

Number of customers: 2438


In [98]:
rfm_recency = []
rfm_frequency = []
rfm_monetary = []

In [94]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

analysis_date = orders["order_date"].max()

print("Analysis date:", analysis_date)

Analysis date: 2024-12-31 23:58:14


In [95]:
analysis_date = np.datetime64(analysis_date)

In [99]:
for customer in unique_customers:

    mask = customer_ids == customer

    customer_dates = sales["order_date"].to_numpy()[mask]
    customer_orders = sales["order_id"].to_numpy()[mask]
    customer_revenue = sales["revenue"].to_numpy()[mask]

    last_purchase = np.max(customer_dates)

    recency = (analysis_date - last_purchase).astype("timedelta64[D]").astype(int)

    frequency = len(np.unique(customer_orders))

    monetary = np.sum(customer_revenue)

    rfm_recency.append(recency)
    rfm_frequency.append(frequency)
    rfm_monetary.append(monetary)

In [100]:
rfm_recency = np.array(rfm_recency)
rfm_frequency = np.array(rfm_frequency)
rfm_monetary = np.array(rfm_monetary)

### RFM scoring

In [101]:
r_thresholds = np.percentile(
    rfm_recency,
    [20, 40, 60, 80]
)

f_thresholds = np.percentile(
    rfm_frequency,
    [20, 40, 60, 80]
)

m_thresholds = np.percentile(
    rfm_monetary,
    [20, 40, 60, 80]
)

### Assign the scores

Frequency

In [102]:
f_score = (
    np.searchsorted(
        f_thresholds,
        rfm_frequency,
        side="right"
    ) + 1
)

Monetary

In [103]:
m_score = (
    np.searchsorted(
        m_thresholds,
        rfm_monetary,
        side="right"
    ) + 1
)

Recency

In [104]:
r_score = 5 - np.searchsorted(
    r_thresholds,
    rfm_recency,
    side="right"
)

### Create our segmentation score

combine them into a single segmentation score you define

### $RFM$ Score $= R + F + M$

In [105]:
rfm_score = r_score + f_score + m_score

In [106]:
rfm_result = pd.DataFrame({
    "customer_id": unique_customers,
    "recency": rfm_recency,
    "frequency": rfm_frequency,
    "monetary": rfm_monetary,
    "R_score": r_score,
    "F_score": f_score,
    "M_score": m_score,
    "RFM_score": rfm_score
})

In [107]:
rfm_result.head(10)

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,1,372,3,2368.55,1,3,2,6
1,2,79,4,4325.62,4,4,3,11
2,3,279,6,12763.42,2,5,5,12
3,4,780,1,24.70,1,1,1,3
4,5,72,2,4238.67,4,2,3,9
5,6,228,2,2289.56,2,2,2,6
6,7,153,1,1701.48,3,1,1,5
7,8,279,2,1679.86,2,2,1,5
8,9,88,3,4036.27,4,3,3,10
9,10,79,2,1779.07,4,2,2,8


In [108]:
check_frequency = (
    sales.groupby("customer_id")["order_id"]
    .nunique()
)

### 2. Product Similarity & Recommmendation

In [109]:
sales.columns.tolist()

['order_item_id',
 'order_id',
 'product_id',
 'quantity',
 'unit_price',
 'discount',
 'is_return',
 'revenue',
 'order_date',
 'category',
 'month',
 'customer_id']

##### $ X_{ij} =$ quantity of product $j$ purchased by customer $i$

In [110]:
recommendation_sales = sales[
    sales["quantity"] > 0
].copy()

In [111]:
recommendation_sales

,order_item_id,order_id,product_id,quantity,unit_price,discount,is_return,revenue,order_date,category,month,customer_id
0,1,1,272,3,5.69,0.15,False,17.07,2022-12-07 09:41:23,Books & Media,2022-12,940
2,3,1,273,1,560.24,0.00,False,560.24,2022-12-07 09:41:23,Books & Media,2022-12,940
3,4,2,11,1,216.27,0.00,False,216.27,2024-06-30 10:09:52,Electronics,2024-06,706
4,5,2,239,3,517.80,0.15,False,1553.40,2024-06-30 10:09:52,Beauty & Health,2024-06,706
5,6,3,58,1,87.55,0.00,False,87.55,2022-04-21 08:21:52,Home & Kitchen,2022-04,520
...,...,...,...,...,...,...,...,...,...,...,...,...
20357,20358,8999,221,3,704.75,0.10,False,2114.25,2023-07-09 01:36:58,Beauty & Health,2023-07,1544
20358,20359,8999,264,1,16.20,0.00,False,16.20,2023-07-09 01:36:58,Books & Media,2023-07,1544
20359,20360,8999,192,1,697.79,0.00,False,697.79,2023-07-09 01:36:58,Sports & Outdoors,2023-07,1544
20360,20361,9000,191,1,457.70,0.00,False,457.70,2022-07-16 12:26:27,Sports & Outdoors,2022-07,2407


In [112]:
purchase_matrix_df = recommendation_sales.pivot_table(
    index="product_id",
    columns="customer_id",
    values="quantity",
    aggfunc="sum",
    fill_value=0
)

In [113]:
purchase_matrix_df

customer_id,1,2,3,4,5,6,7,8,9,10,...,2490,2491,2492,2493,2494,2495,2496,2497,2499,2500
product_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
297,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
298,0,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [114]:
product_matrix = purchase_matrix_df.to_numpy(dtype=float)

In [115]:
product_matrix.shape

(300, 2435)

#### cosine similarity($A,B$) $= \frac{A \times B}{|A||B|}$

where <br>
#### Dot product <br>
#### $A \times B = \sum_i A_{i}B_{i}$

#### Vector norm

#### $|A| = \sqrt{\sum_{i} A_{i}^{2}}$

Calculate similarity using NumPy

In [116]:
dot_products = product_matrix @ product_matrix.T

In [117]:
dot_products

array([[129.,   1.,   5., ...,   2.,   9.,   5.],
       [  1., 206.,   9., ...,   2.,   8.,   4.],
       [  5.,   9., 132., ...,   2.,   4.,   0.],
       ...,
       [  2.,   2.,   2., ..., 137.,   4.,   3.],
       [  9.,   8.,   4., ...,   4., 154.,  10.],
       [  5.,   4.,   0., ...,   3.,  10., 157.]], shape=(300, 300))

In [118]:
norms = np.linalg.norm(
    product_matrix,
    axis=1
)

In [119]:
norms

array([11.35781669, 14.35270009, 11.48912529, 12.68857754, 11.78982612,
       11.18033989, 10.67707825, 12.20655562, 13.26649916, 10.67707825,
       14.56021978, 14.49137675, 10.09950494,  9.74679434, 14.38749457,
       13.49073756, 11.13552873, 11.09053651, 12.32882801, 10.39230485,
       13.49073756, 12.56980509, 11.83215957, 11.91637529, 13.49073756,
       10.19803903, 14.31782106, 11.87434209, 13.26649916, 11.48912529,
       10.67707825, 13.41640786, 12.12435565, 11.78982612, 11.13552873,
       11.18033989, 10.86278049, 10.63014581, 11.18033989, 11.87434209,
       11.87434209,  9.59166305, 11.87434209, 11.35781669, 11.70469991,
       11.44552314, 12.36931688, 11.09053651, 11.87434209, 10.04987562,
       11.61895004, 12.20655562, 11.48912529, 11.09053651, 12.36931688,
       11.18033989, 11.35781669, 12.12435565, 12.76714533, 11.44552314,
       11.09053651, 12.76714533, 11.78982612, 13.07669683, 11.        ,
       13.11487705, 11.3137085 , 13.07669683, 11.40175425, 10.90

In [120]:
similarity_matrix = (
    dot_products /
    (norms[:, None] * norms[None, :])
)

In [121]:
similarity_matrix

array([[1.        , 0.00613439, 0.03831671, ..., 0.0150444 , 0.06385388,
        0.03513382],
       [0.00613439, 1.        , 0.05457854, ..., 0.01190518, 0.04491548,
        0.02224214],
       [0.03831671, 0.05457854, 1.        , ..., 0.01487246, 0.02805515,
        0.        ],
       ...,
       [0.0150444 , 0.01190518, 0.01487246, ..., 1.        , 0.02753844,
        0.02045555],
       [0.06385388, 0.04491548, 0.02805515, ..., 0.02753844, 1.        ,
        0.06431167],
       [0.03513382, 0.02224214, 0.        , ..., 0.02045555, 0.06431167,
        1.        ]], shape=(300, 300))

#### Deal with products with zero purchases

In [122]:
denominator = norms[:, None] * norms[None, :]

similarity_matrix = np.divide(
    dot_products,
    denominator,
    out=np.zeros_like(dot_products),
    where=denominator != 0
)

#### Verify the result

In [123]:
similarity_matrix.shape

(300, 300)

In [124]:
np.diag(similarity_matrix)[:10]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

#### Check symmetry

$sim(A,B) = sim(B,A)$

In [125]:
np.allclose(
    similarity_matrix,
    similarity_matrix.T
)

True

Recommend 3 products

In [126]:
sample_customers = (
    recommendation_sales["customer_id"]
    .drop_duplicates()
    .head(5)
    .to_numpy()
)

sample_customers

array([ 940,  706,  520, 2249, 2240])

Create a product index

In [127]:
product_ids = purchase_matrix_df.index.to_numpy()

product_index = {
    product_id: i
    for i, product_id in enumerate(product_ids)
}

Generate recommendations

In [128]:
recommendations = {}

for customer in sample_customers:

    customer_rows = recommendation_sales[
        recommendation_sales["customer_id"] == customer
    ]

    purchased_products = customer_rows[
        "product_id"
    ].unique()

    purchased_indices = [
        product_index[p]
        for p in purchased_products
        if p in product_index
    ]

    # Add similarity scores from purchased products
    scores = similarity_matrix[purchased_indices].sum(axis=0)

    # Don't recommend products already purchased
    scores[purchased_indices] = -np.inf

    # Get top 3
    top_indices = np.argsort(scores)[-3:][::-1]

    recommendations[customer] = product_ids[top_indices]

In [129]:
recommendations

{np.int64(940): array([ 55,  96, 102]),
 np.int64(706): array([158,  92, 190]),
 np.int64(520): array([296, 131, 264]),
 np.int64(2249): array([275,  65,  86]),
 np.int64(2240): array([185, 202, 220])}

Make the output readable

In [130]:
recommendation_rows = []

for customer, products_rec in recommendations.items():

    for rank, product in enumerate(products_rec, start=1):

        recommendation_rows.append({
            "customer_id": customer,
            "rank": rank,
            "recommended_product": product
        })

recommendation_df = pd.DataFrame(
    recommendation_rows
)

recommendation_df

,customer_id,rank,recommended_product
0,940,1,55
1,940,2,96
2,940,3,102
3,706,1,158
4,706,2,92
5,706,3,190
6,520,1,296
7,520,2,131
8,520,3,264
9,2249,1,275


In [131]:
np.diag(similarity_matrix)[:10]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [132]:
np.allclose(
    similarity_matrix,
    similarity_matrix.T
)

True

### 3. Regression via Normal Equation

$y = \beta_{0} + \beta_{1}x$

where <br>
- $x$ = month index
- $y$ = monthly revenue
- $\beta_{0}$ = intercept
- $\beta_{1}$ = revenue change per month

Prepare monthly revenue

In [133]:
monthly_revenue.head()

order_date
2022-01-31    300743.70
2022-02-28    167961.02
2022-03-31    232057.39
2022-04-30    178805.73
2022-05-31    296399.88
Freq: ME, Name: revenue, dtype: float64

In [134]:
y = monthly_revenue.to_numpy(dtype=float)

In [135]:
print(y)
print("Number of observations:", len(y))

[300743.7  167961.02 232057.39 178805.73 296399.88 203081.94 364664.56
 227561.66 259651.07 325780.9  276018.42 285640.52 276829.82 280914.21
 313134.84 244362.11 475339.93 369491.91 384002.74 295945.69 458585.24
 349623.86 463248.9  452961.98 476364.82 424839.33 456547.69 451799.65
 540371.3  517991.09 546455.96 617747.71 521183.93 468001.8  640420.72
 506203.53]
Number of observations: 36


Create the month index

In [136]:
x = np.arange(1, len(y) + 1, dtype=float)

In [137]:
x

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 34., 35., 36.])

build the design matrix **$X$**

In [138]:
X = np.column_stack([
    np.ones(len(x)),
    x
])

In [139]:
print(X[:5])

[[1. 1.]
 [1. 2.]
 [1. 3.]
 [1. 4.]
 [1. 5.]]


Apply the normal equation

The formula is $\beta = (X^{T}X)^{-1}X^{T}y$

In [140]:
XTX = X.T @ X
XTy = X.T @ y

beta = np.linalg.inv(XTX) @ XTy

In [141]:
beta

array([183118.80320635,  10598.2862381 ])

Look at the coefficients

In [142]:
beta_0 = beta[0]
beta_1 = beta[1]

print("Intercept:", beta_0)
print("Slope:", beta_1)

Intercept: 183118.80320634914
Slope: 10598.286238095248


Calculate predicted revenue

$\hat{y} = X\beta$

In [143]:
y_pred = X @ beta

In [144]:
print(y_pred[:10])

[193717.08944444 204315.37568254 214913.66192063 225511.94815873
 236110.23439683 246708.52063492 257306.80687302 267905.09311111
 278503.37934921 289101.6655873 ]


Calculate $R^2$ manually

first calculate the residual sum of squares

use <br>
$SS_{res} = \sum(y_{i}-\hat{y}_{i})^2 $

In [145]:
ss_res = np.sum(
    (y - y_pred) ** 2
)

Then calculate the toal sum of squares

$SS_{tot} = \sum(y_{i} - \hat{y})^2$

where: <br>
<br>
$\hat{y} = \frac{1}{n} \sum y_{i}$

In [146]:
y_mean = np.mean(y)

ss_tot = np.sum(
    (y - y_mean) ** 2
)

Finally:

$R^{2} = 1 - \frac{SS_{res}}{SS{tot}}$

In [147]:
r2 = 1 - (ss_res / ss_tot)

In [148]:
print("R²:", r2)

R²: 0.787433842865173


Sanity Check

In [149]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [150]:
from sklearn.metrics import r2_score

r2_check = r2_score(y, y_pred)

print("Manual R²:", r2)
print("Library R² check:", r2_check)

Manual R²: 0.787433842865173
Library R² check: 0.787433842865173


In [151]:
residuals = y - y_pred

print("Mean residual:", np.mean(residuals))

Mean residual: -1.1479844235711628e-10


results

In [152]:
regression_result = pd.DataFrame({
    "month_index": x,
    "actual_revenue": y,
    "predicted_revenue": y_pred,
    "residual": residuals
})

regression_result.head()

,month_index,actual_revenue,predicted_revenue,residual
0,1.0,300743.70,193717.089444,107026.610556
1,2.0,167961.02,204315.375683,-36354.355683
2,3.0,232057.39,214913.661921,17143.728079
3,4.0,178805.73,225511.948159,-46706.218159
4,5.0,296399.88,236110.234397,60289.645603


### 4. Monte Carlo Simulation

In [154]:
import pandas as pd
import numpy as np

catalog_df = pd.read_csv("../data/processed/product_catalog_clean.csv")

print(catalog_df.columns.tolist())

['product_id', 'name', 'category', 'unit_price', 'cost', 'stock']


In [155]:
catalog[[
    "product_id",
    "name",
    "stock"
]].head(20)

,product_id,name,stock
0,90,Group Furniture,393
1,242,Huge Personal Care,281
2,230,Fight Supplement,261
3,213,Star Supplement,359
4,245,Throughout Personal Care,354
5,125,Increase Women,276
6,295,Among Media,365
7,118,During Men,82
8,103,Carry Women,166
9,298,Wait Media,281


Understand our demand model (model assumption)

$D \sim N(\mu, \sigma^2)$

where <br>
- $\mu$ = historical average demand
- $\sigma$ = historical demand standard deviation

Get historical demand

In [156]:
print(sales.columns.tolist())

['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount', 'is_return', 'revenue', 'order_date', 'category', 'month', 'customer_id']


In [157]:
sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    errors="coerce"
)

In [158]:
sales_mc = sales[
    sales["order_date"].notna()
].copy()

In [159]:
sales_mc = sales_mc[
    sales_mc["quantity"] > 0
].copy()

Create monthly demand

In [160]:
sales_mc["month"] = sales_mc["order_date"].dt.to_period("M")

In [161]:
monthly_demand = (
    sales_mc
    .groupby(["product_id", "month"])["quantity"]
    .sum()
)

In [162]:
monthly_demand.head(10)

product_id  month  
1           2022-02    1
            2022-04    1
            2022-05    1
            2022-06    2
            2022-07    3
            2022-08    2
            2022-09    4
            2022-11    2
            2023-01    1
            2023-03    5
Name: quantity, dtype: int64

choose 3 products

In [163]:
total_demand = (
    sales_mc
    .groupby("product_id")["quantity"]
    .sum()
)

In [164]:
chosen_products = (
    total_demand
    .nlargest(3)
    .index
    .to_numpy()
)

In [165]:
print("Chosen products:")
print(chosen_products)

Chosen products:
[ 15  72 199]


Check that these products exist in the catalog

In [166]:
catalog_df

,product_id,name,category,unit_price,cost,stock
0,90,Group Furniture,Home & Kitchen,220.234977,99.04,393
1,242,Huge Personal Care,Beauty & Health,166.720823,79.41,281
2,230,Fight Supplement,Beauty & Health,853.221347,362.81,261
3,213,Star Supplement,Beauty & Health,113.997777,82.05,359
4,245,Throughout Personal Care,Beauty & Health,11782.864610,104.23,354
...,...,...,...,...,...,...
262,9007,Leader Prototype,Sports & Outdoors,107.980000,61.73,24
263,9008,Sign Prototype,Electronics,274.140000,91.39,4
264,9009,Trial Prototype,Electronics,141.540000,132.01,49
265,9010,Attention Prototype,Beauty & Health,157.720000,14.75,35


In [167]:
for product_id in chosen_products:
    exists = product_id in catalog_df["product_id"].values
    print(product_id, "exists in catalog:", exists)

15 exists in catalog: True
72 exists in catalog: True
199 exists in catalog: True


Create the stock lookup

In [168]:
stock_lookup = catalog_df.set_index("product_id")["stock"]

In [169]:
print(type(stock_lookup))

<class 'pandas.Series'>


In [170]:
for product_id in chosen_products:
    stock_value = int(stock_lookup.loc[product_id])
    print(product_id, "Stock:", stock_value)

15 Stock: 50
72 Stock: 142
199 Stock: 62


Prepare the simulation

In [171]:
N_TRIALS = 10000

rng = np.random.default_rng(42)

In [172]:
print(rng)

Generator(PCG64)


Run the Monte Carlo simulation

In [173]:
simulation_results = []

for product_id in chosen_products:

    # Historical monthly demand for this product
    product_monthly_demand = monthly_demand[
        monthly_demand.index.get_level_values("product_id") == product_id
    ].to_numpy(dtype=float)

    # Historical demand statistics
    mean_demand = np.mean(product_monthly_demand)
    std_demand = np.std(
        product_monthly_demand,
        ddof=1
    )

    # Current inventory
    stock_value = int(
        stock_lookup.loc[product_id]
    )

    # Generate 10,000 possible future demand values
    simulated_demand = rng.normal(
        loc=mean_demand,
        scale=std_demand,
        size=N_TRIALS
    )

    # Stockout occurs when demand exceeds available stock
    stockout = simulated_demand > stock_value

    # Estimated probability
    stockout_probability = np.mean(stockout)

    # Standard error
    standard_error = np.sqrt(
        stockout_probability
        * (1 - stockout_probability)
        / N_TRIALS
    )

    # 95% confidence interval
    ci_lower = (
        stockout_probability
        - 1.96 * standard_error
    )

    ci_upper = (
        stockout_probability
        + 1.96 * standard_error
    )

    # Keep CI within 0–1
    ci_lower = max(0, ci_lower)
    ci_upper = min(1, ci_upper)

    simulation_results.append({
        "product_id": product_id,
        "stock": stock_value,
        "mean_monthly_demand": mean_demand,
        "std_monthly_demand": std_demand,
        "stockout_probability": stockout_probability,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper
    })

Create the results table

In [174]:
monte_carlo_result = pd.DataFrame(
    simulation_results
)

monte_carlo_result

,product_id,stock,mean_monthly_demand,std_monthly_demand,stockout_probability,ci_lower,ci_upper
0,15,50,3.606061,2.090744,0.0,0,0.0
1,72,142,3.966667,2.511811,0.0,0,0.0
2,199,62,3.645161,1.703886,0.0,0,0.0


Convert the probabilities to percantages

In [175]:
monte_carlo_result["stockout_probability_pct"] = (
    monte_carlo_result["stockout_probability"] * 100
)

monte_carlo_result["ci_lower_pct"] = (
    monte_carlo_result["ci_lower"] * 100
)

monte_carlo_result["ci_upper_pct"] = (
    monte_carlo_result["ci_upper"] * 100
)

In [176]:
monte_carlo_result[
    [
        "product_id",
        "stock",
        "mean_monthly_demand",
        "std_monthly_demand",
        "stockout_probability_pct",
        "ci_lower_pct",
        "ci_upper_pct"
    ]
]

,product_id,stock,mean_monthly_demand,std_monthly_demand,stockout_probability_pct,ci_lower_pct,ci_upper_pct
0,15,50,3.606061,2.090744,0.0,0,0.0
1,72,142,3.966667,2.511811,0.0,0,0.0
2,199,62,3.645161,1.703886,0.0,0,0.0


Sanity Check

Test 1: Massive inventory

In [177]:
test_demand = rng.normal(
    loc=100,
    scale=20,
    size=10000
)

test_stock = 1_000_000

test_probability = np.mean(
    test_demand > test_stock
)

print("Stockout probability with huge stock:", test_probability)

Stockout probability with huge stock: 0.0


Test 2: Zero inventory

In [178]:
test_stock = 0

test_probability = np.mean(
    test_demand > test_stock
)

print("Stockout probability with zero stock:", test_probability)

Stockout probability with zero stock: 1.0
